# ⚙️ Project 11.3 — Factory Parameter Optimizer
**DSA for Mechatronics · Week 11 — Searching Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, bisect, time
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A factory engineer needs to tune parameters that cannot be solved with a formula
but can be optimised by **binary search on the answer space**.

Key insight: if we can write a function `feasible(x) → bool` that checks
whether a given parameter value x works, AND feasible() is monotone
(once True, always True for larger x), then we can binary search for the
minimum x that satisfies the constraint.

Three real factory problems:
1. **Integer square root** — largest integer n where n² ≤ x (sensor ADC range)
2. **Minimum conveyor speed** — minimum rate to finish N batches within T time slots
3. **Minimum robot capacity** — minimum load capacity so a robot completes all
   deliveries within D days


## Step 1 — Integer square root (binary search on answer)

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
ADC_VALUES   = [random.randint(1000, 90000) for _ in range(6)]
BATCH_SIZES  = sorted([random.randint(5, 50) for _ in range(random.randint(8, 14))])
TIME_SLOTS   = random.randint(len(BATCH_SIZES), len(BATCH_SIZES) * 3)
PACKAGES     = sorted([random.randint(10, 80) for _ in range(random.randint(8, 15))])
DAYS_LIMIT   = random.randint(len(PACKAGES), len(PACKAGES) * 2)

def integer_sqrt(x):
    """
    Find floor(sqrt(x)) without using math.sqrt.

    Binary search on answer space [0, x]:
      feasible(n) = (n * n <= x)
    Find the LARGEST n where feasible(n) is True.

    Template: find rightmost True in [lo, hi] where condition is monotone
      True True True ... True FALSE FALSE FALSE
      We want the last True → when feasible, record result and go RIGHT.
    O(log x) steps.
    """
    if x < 2: return x
    lo, hi  = 1, x // 2 + 1
    result  = 0
    steps   = 0
    while lo <= hi:
        steps += 1
        mid = lo + (hi - lo) // 2
        if mid * mid <= x:
            result = mid        # feasible — record and try bigger
            lo = mid + 1
        else:
            hi = mid - 1        # too large — go smaller
    return result, steps

print(f"Integer square root (binary search on answer):")
print(f"  {'x':>8}  {'isqrt(x)':>10}  {'x+1 check':>11}  {'Steps':>7}  {'Python int(x**0.5)':>20}")
print(f"  {'─'*8}  {'─'*10}  {'─'*11}  {'─'*7}  {'─'*20}")
for x in ADC_VALUES:
    res, st = integer_sqrt(x)
    import math
    true_val = math.isqrt(x)
    ok  = res == true_val
    print(f"  {x:>8}  {res:>10}  {(res+1)**2:>11}  {st:>7}  {true_val:>20}  {'✅' if ok else '❌'}")


## Step 2 — Minimum conveyor speed (Koko-style)

In [ ]:
def can_finish_batches(batches, speed, slots):
    """
    Check if processing all batches at given speed finishes within slots time slots.
    Each slot: process up to `speed` units. If batch > speed, it takes
    ceil(batch / speed) slots. Total slots used must be <= slots.
    """
    import math
    total = sum(math.ceil(b / speed) for b in batches)
    return total <= slots

def min_conveyor_speed(batches, slots):
    """
    Binary search on answer: find MINIMUM speed that allows finishing all batches
    within the given number of time slots.

    Search space: [1, max(batches)]
      - At speed=1: each unit takes 1 slot — need sum(batches) slots total
      - At speed=max(batches): each batch takes exactly 1 slot
    Monotone: if speed s works, any speed > s also works.
    → Binary search for leftmost True.
    O(n log(max_batch)) where n = len(batches).
    """
    lo, hi   = 1, max(batches)
    result   = hi
    steps    = 0
    while lo <= hi:
        steps += 1
        mid = lo + (hi - lo) // 2
        if can_finish_batches(batches, mid, slots):
            result = mid      # feasible — try smaller speed
            hi = mid - 1
        else:
            lo = mid + 1      # too slow — need faster
    return result, steps

speed, speed_steps = min_conveyor_speed(BATCH_SIZES, TIME_SLOTS)

# Brute-force verification
import math
bf_speed = next(s for s in range(1, max(BATCH_SIZES)+1)
                if sum(math.ceil(b/s) for b in BATCH_SIZES) <= TIME_SLOTS)

print(f"Minimum conveyor speed (Koko-style binary search on answer):")
print(f"  Batch sizes  : {BATCH_SIZES}")
print(f"  Time slots   : {TIME_SLOTS}")
print(f"  Search space : [1, {max(BATCH_SIZES)}]")
print()
print(f"  Minimum speed found : {speed} units/slot")
print(f"  Brute-force check   : {bf_speed}")
print(f"  Correct             : {'✅ YES' if speed == bf_speed else '❌ NO'}")
print(f"  Steps               : {speed_steps}  (brute force: {max(BATCH_SIZES)} checks)")
print()
# Verify the boundary
slots_at_speed     = sum(math.ceil(b / speed) for b in BATCH_SIZES)
slots_at_speed_m1  = sum(math.ceil(b / (speed - 1)) for b in BATCH_SIZES) if speed > 1 else float('inf')
print(f"  Slots at speed {speed}   : {slots_at_speed} ≤ {TIME_SLOTS}  {'✅' if slots_at_speed <= TIME_SLOTS else '❌'}")
if speed > 1:
    print(f"  Slots at speed {speed-1}   : {slots_at_speed_m1} > {TIME_SLOTS}  "
          f"{'✅ (confirms minimum)' if slots_at_speed_m1 > TIME_SLOTS else '❌'}")


## Step 3 — Minimum robot load capacity (Ship packages problem)

In [ ]:
def can_deliver(packages, capacity, days):
    """
    Check if a robot with given weight capacity can deliver all packages
    within the given number of days.
    Load greedily each day: add packages in order until capacity exceeded.
    """
    day_load = 0
    days_used = 1
    for p in packages:
        if p > capacity:
            return False           # single package too heavy — impossible
        if day_load + p > capacity:
            days_used += 1         # start new day
            day_load = 0
        day_load += p
    return days_used <= days

def min_robot_capacity(packages, days):
    """
    Binary search on answer: minimum capacity so robot delivers all packages
    in at most `days` days, keeping package order (conveyor belt model).

    Search space: [max(packages), sum(packages)]
      - Lower bound: must carry heaviest single package
      - Upper bound: carry everything in one day
    Monotone: if capacity c works, any c' > c also works.
    O(n log(sum)) steps.
    """
    lo, hi  = max(packages), sum(packages)
    result  = hi
    steps   = 0
    while lo <= hi:
        steps += 1
        mid = lo + (hi - lo) // 2
        if can_deliver(packages, mid, days):
            result = mid       # feasible — try smaller capacity
            hi = mid - 1
        else:
            lo = mid + 1       # not enough — increase capacity
    return result, steps

capacity, cap_steps = min_robot_capacity(PACKAGES, DAYS_LIMIT)
# Brute force verify
bf_cap = next(c for c in range(max(PACKAGES), sum(PACKAGES)+1)
              if can_deliver(PACKAGES, c, DAYS_LIMIT))

print(f"Minimum robot delivery capacity:")
print(f"  Package weights : {PACKAGES}")
print(f"  Days allowed    : {DAYS_LIMIT}")
print(f"  Search space    : [{max(PACKAGES)}, {sum(PACKAGES)}]")
print()
print(f"  Minimum capacity found : {capacity} kg")
print(f"  Brute-force check      : {bf_cap} kg")
print(f"  Correct                : {'✅ YES' if capacity == bf_cap else '❌ NO'}")
print(f"  Steps                  : {cap_steps}  (search space: {sum(PACKAGES) - max(PACKAGES) + 1})")
print()
# Show day-by-day delivery plan at minimum capacity
print(f"  Day-by-day delivery plan at capacity {capacity}:")
day_load = 0; day = 1; day_pkgs = []
plan = []
for p in PACKAGES:
    if day_load + p > capacity:
        plan.append((day, list(day_pkgs), day_load))
        day += 1; day_load = 0; day_pkgs = []
    day_load += p; day_pkgs.append(p)
plan.append((day, list(day_pkgs), day_load))
for d, pkgs, load in plan:
    print(f"    Day {d}: {pkgs}  (load={load}/{capacity})")


## Step 4 — Binary search template summary + pitfall demo

In [ ]:
# The universal binary search template: find leftmost True in monotone bool array
# ─────────────────────────────────────────────────────────────────────────────
# TEMPLATE A — find leftmost index where condition(arr[i]) is True
# ─────────────────────────────────────────────────────────────────────────────
def binary_search_leftmost(arr, condition):
    """
    Find the smallest index i such that condition(arr[i]) is True.
    Assumes arr is sorted such that condition is False...False True...True.
    Returns -1 if no such index exists.
    """
    lo, hi = 0, len(arr) - 1
    result = -1
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        if condition(arr[mid]):
            result = mid
            hi = mid - 1     # keep searching left for earlier True
        else:
            lo = mid + 1
    return result

# ─────────────────────────────────────────────────────────────────────────────
# PITFALL DEMO: off-by-one error in loop termination
# ─────────────────────────────────────────────────────────────────────────────
def binary_search_infinite_loop_demo(arr, target):
    """Demonstrates the pitfall of lo < hi without mid+1 → infinite loop."""
    lo, hi = 0, len(arr) - 1
    max_iters = 0
    # WRONG: when lo+1 == hi and arr[lo] < target, mid == lo, lo never advances
    # We simulate only 20 iterations to show the loop gets stuck
    while lo < hi and max_iters < 20:
        max_iters += 1
        mid = lo + (hi - lo) // 2   # mid == lo when hi = lo+1
        if arr[mid] < target:
            lo = mid                 # BUG: should be mid + 1
        else:
            hi = mid
    return lo, max_iters, "⚠ STUCK" if max_iters == 20 else "✅ ok"

# Demonstrate templates on all three problems
print(f"Binary search template summary:")
print()
print(f"  Problem 1 — Integer sqrt of {ADC_VALUES[0]}:")
res_t, _ = integer_sqrt(ADC_VALUES[0])
# Using leftmost template: find largest n where n²<=x  ↔  find leftmost False
arr_sq = list(range(0, ADC_VALUES[0]//2 + 2))
first_too_big = binary_search_leftmost(arr_sq, lambda n: n*n > ADC_VALUES[0])
template_sqrt = first_too_big - 1 if first_too_big > 0 else 0
print(f"    Direct BS result     : {res_t}")
print(f"    Template (leftmost)  : {template_sqrt}  {'✅' if template_sqrt == res_t else '❌'}")

print()
print(f"  Problem 2 — Conveyor speed (feasibility boundary):")
speeds = list(range(1, max(BATCH_SIZES)+1))
first_ok_speed = binary_search_leftmost(speeds, lambda s: can_finish_batches(BATCH_SIZES, s, TIME_SLOTS))
print(f"    Direct BS result     : {speed}")
print(f"    Template (leftmost)  : {first_ok_speed}  {'✅' if first_ok_speed == speed else '❌'}")

print()
print(f"  Pitfall demo — infinite loop (lo < hi without mid+1):")
demo_arr = sorted(BATCH_SIZES)
lo_demo, iters, verdict = binary_search_infinite_loop_demo(demo_arr, demo_arr[-1])
print(f"    Searching for {demo_arr[-1]} in {demo_arr}")
print(f"    Iterations : {iters}  {verdict}")
print(f"    Fix        : change 'lo = mid' to 'lo = mid + 1'")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " FACTORY PARAMETER OPTIMIZER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  INTEGER SQRT" + " "*(W-14) + "║")
print(f"║  {'ADC values tested':<28}: {len(ADC_VALUES):<26}║")
_, st0 = integer_sqrt(ADC_VALUES[0])
print(f"║  {'Steps (sample)':<28}: {st0} for x={ADC_VALUES[0]}{'':<10}║")
print("╠" + "═"*W + "╣")
print("║  CONVEYOR SPEED (Koko)" + " "*(W-23) + "║")
print(f"║  {'Batches':<28}: {len(BATCH_SIZES):<26}║")
print(f"║  {'Time slots':<28}: {TIME_SLOTS:<26}║")
print(f"║  {'Minimum speed found':<28}: {speed} units/slot{'':<14}║")
print(f"║  {'Binary search steps':<28}: {speed_steps}  (space 1..{max(BATCH_SIZES)}){'':<5}║")
print(f"║  {'Correct':<28}: {'YES ✅' if speed == bf_speed else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  ROBOT CAPACITY (Ship)" + " "*(W-23) + "║")
print(f"║  {'Packages':<28}: {len(PACKAGES):<26}║")
print(f"║  {'Days allowed':<28}: {DAYS_LIMIT:<26}║")
print(f"║  {'Minimum capacity found':<28}: {capacity} kg{'':<21}║")
print(f"║  {'Binary search steps':<28}: {cap_steps}  (space {max(PACKAGES)}..{sum(PACKAGES)}){'':<3}║")
print(f"║  {'Correct':<28}: {'YES ✅' if capacity == bf_cap else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about searching in this context?

*Your answer here:*

---

**Q2 — Why is binary search O(log n) and not O(n)?**
Explain in your own words why halving the search space each step leads to logarithmic complexity,
and give one example from your results that illustrates this.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
